# Chapter 1 - Lab 1: API Calls - Open and closed-source LLMs (customer-care edition)

**Goal**: learn the most basic building block of every agent: a single call to a Large Language Model (LLM).

We ask four model families (OpenAI, Google Gemini, Anthropic Claude, and an open-weights Llama model)
to perform the **same customer-care task**: summarise a support-call transcript into a structured
JSON note that a service manager can act on. We then compare with a reasoning model.

*Finance version of this lab*: summarising Apple's earnings call for a portfolio manager.
*Customer-care version*: summarising a customer's escalation call for a service manager.
The skill is the same: turn long unstructured text into a short, structured, decision-useful record.

## The synthetic support-call transcript

NovaMart is a **fictional** Indian electronics and appliance retailer used across all chapters.
Priya Sharma (customer `C1001`, Platinum tier) has a refrigerator that stopped cooling.
Two repair visits failed; she has threatened a social-media post and a consumer-court notice.

In [ ]:
# The following support-call transcript is synthetic (NovaMart is a fictional retailer):

transcript_support_call = """
# Synthetic Customer Support Call Transcript - NovaMart Care (Example)

Call ID: CALL-2026-10-26-0417 | Channel: Phone (escalation desk) | Duration: 11 min 42 s
Customer: Priya Sharma (Platinum member since 2019, customer ID C1001)
Agent: Arjun Mehta, Senior Care Specialist

Arjun: Good afternoon, thank you for calling NovaMart Care, this is Arjun. May I have your customer ID or registered mobile number?

Priya: It's C1001. I'm calling for the fourth time about the FrostLine refrigerator I bought last month. Order ORD-77812.

Arjun: I'm sorry to hear that, Ms Sharma. I can see the order: FrostLine 340-litre, delivered on 18 September, Rs 62,990. I can also see three earlier tickets. Could you tell me what is happening now?

Priya: It stopped cooling on 9 October. Your technician came on the 12th, replaced a sensor, it worked for two days and stopped again. Second visit on the 19th, he said the compressor relay needs a part that is "not in stock". It is now more than two weeks. I have thrown away food worth at least four thousand rupees and I have elderly parents at home.

Arjun: I completely understand, and I apologise. That is not the experience we want for you, especially as one of our Platinum members.

Priya: I have been a loyal customer for seven years. My TV, AC, washing machine, everything is from NovaMart. But honestly, if this is not fixed I will move to another store. I have already written a post on Twitter and I'm thinking of sending a consumer court notice.

Arjun: Let me check the options. Under our warranty, if the same defect persists after two repair attempts, the product qualifies for a replacement. Two repairs have failed, so you are eligible. I can raise a replacement request right now.

Priya: How long will that take?

Arjun: For a large appliance the standard replacement time is 5 to 7 working days. Given what you have been through, I am requesting priority dispatch from our Pune warehouse. I am checking stock... yes, the same model is in stock. I can commit to delivery and installation within 72 hours, by Thursday 29 October.

Priya: And the food I lost? And all my time?

Arjun: I'll add a goodwill credit of Rs 3,000 to your NovaMart wallet today, and I'll waive the installation charge on the new unit. I'm also escalating your case to my supervisor, Ms Kavita Rao, who will call you tomorrow before 12 noon to confirm the dispatch.

Priya: Fine. But if it doesn't arrive by Thursday, I'm going to the consumer forum.

Arjun: Understood. I've noted that. Your replacement reference is RPL-90318. You'll get an SMS and WhatsApp confirmation within the hour. Is there anything else I can help you with?

Priya: No. Just make sure it happens this time.

Arjun: It will. Thank you for your patience, Ms Sharma.

--- Agent wrap-up notes (after-call work) ---
Disposition: Replacement approved (repeat-failure rule: 2 failed repairs). Goodwill credit Rs 3,000 (supervisor approval pending).
Priority dispatch from Pune warehouse, ETA 29-Oct. Supervisor callback 27-Oct before 12:00.
Risk flags: public social-media post, consumer-court threat, Platinum member (7-year tenure).
"""

## Closed-source models

### OpenAI (Responses API)

* Create an API key and add a few dollars of credit: https://platform.openai.com/api-keys
* OpenAI offers two APIs: `chat.completions` (older) and `responses` (newer). We use **Responses**.

In [ ]:
%pip install -q openai

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

In [ ]:
import json, os

The **system prompt** tells the model *who it is* and *what shape* the answer must take. The **user prompt** carries the data.

In [ ]:
system_prompt = """You are a customer-experience analyst at NovaMart. Produce a concise,
decision-useful summary of a support call for the service manager. Return strict JSON with keys:
customer, customer_id, order_id, product, issue_summary, timeline[], customer_sentiment,
actions_taken[], commitments[], risks[], recommended_next_step."""

user_prompt = f"""Summarize the following support-call transcript:
{transcript_support_call}"""

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-mini",
    input=[{"role": "system", "content": system_prompt},
           {"role": "user", "content": user_prompt}],
    text={"format": {"type": "json_object"}},   # ask for a JSON object back
)

summary_openai = json.loads(response.output_text)
print(json.dumps(summary_openai, indent=2, ensure_ascii=False))

**What to notice**: `text.format = json_object` makes the model return machine-readable JSON.
A CRM system can store `commitments[]` with due dates and trigger reminders automatically.

### Google Gemini

Gemini models are listed at https://ai.google.dev/gemini-api/docs/models. Create a key in Google AI Studio.

In [ ]:
%pip install -q google-genai

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY first (Colab secret or .env)'

In [ ]:
from google import genai

gclient = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

resp_genai = gclient.models.generate_content(
    model="gemini-2.5-flash",
    contents=system_prompt + "\n\n" + user_prompt,
    config={"response_mime_type": "application/json"},
)
print(resp_genai.text)

# A 503 "model is overloaded" error comes from Google's side: simply retry.

### Anthropic Claude

* Create a key at https://console.anthropic.com/settings/keys and add credits.
* Here we use Claude's **structured outputs**: instead of *asking* for JSON in the prompt, we pass a JSON
  schema in `output_config.format` and the API guarantees the reply matches it.
* `fallbacks="default"` (a beta feature) re-runs the request on another Claude model if the first model
  declines it for safety reasons, so a refusal does not silently break your workflow.

In [ ]:
%pip install -q anthropic

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY first (Colab secret or .env)'

In [ ]:
import anthropic

claude = anthropic.Anthropic()
CLAUDE_MODEL = os.environ.get("CC_ANTHROPIC_MODEL", "claude-opus-5-5")

CALL_SUMMARY_SCHEMA = {
    "type": "object",
    "properties": {
        "customer": {"type": "string"},
        "customer_id": {"type": "string"},
        "order_id": {"type": "string"},
        "product": {"type": "string"},
        "issue_summary": {"type": "string"},
        "timeline": {"type": "array", "items": {"type": "string"}},
        "customer_sentiment": {"type": "string"},
        "actions_taken": {"type": "array", "items": {"type": "string"}},
        "commitments": {"type": "array", "items": {"type": "string"}},
        "risks": {"type": "array", "items": {"type": "string"}},
        "recommended_next_step": {"type": "string"},
    },
    "required": ["customer", "customer_id", "order_id", "product", "issue_summary", "timeline",
                 "customer_sentiment", "actions_taken", "commitments", "risks", "recommended_next_step"],
    "additionalProperties": False,
}

message = claude.beta.messages.create(
    model=CLAUDE_MODEL,
    max_tokens=16000,
    system=system_prompt,
    messages=[{"role": "user", "content": user_prompt}],
    output_config={"format": {"type": "json_schema", "schema": CALL_SUMMARY_SCHEMA}},
    betas=["server-side-fallback-2026-07-01"],
    fallbacks="default",
)

if message.stop_reason == "refusal":
    raise RuntimeError(f"Claude declined the request: {message.stop_details}")
# The response may also contain "thinking" blocks; we only read the text block.
summary_claude = json.loads(next(b.text for b in message.content if b.type == "text"))
print(json.dumps(summary_claude, indent=2, ensure_ascii=False))

## Hugging Face - open-weights model (Llama 3.1 8B Instruct)

Open-weights models run on **your own machine** (here: a Colab GPU). Customer data never leaves your
infrastructure, which matters for privacy regulation such as India's DPDP Act.

Prerequisites: Colab *Runtime -> Change runtime type -> T4 GPU*, a Hugging Face token stored as the
Colab secret `HF_TOKEN`, and accepted access to the gated model card
(https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct). If memory runs out, switch `model_id`
to `meta-llama/Llama-3.2-3B-Instruct`.

In [ ]:
import torch
import transformers

model_id = "meta-llama/Llama-3.1-8B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

# A general-knowledge question first
messages = [{"role": "user", "content": "What does First Contact Resolution (FCR) mean in customer service?"}]
outputs = pipeline(messages, max_new_tokens=256)
print(outputs[0]["generated_text"][-1]["content"])

Now the same summarisation task. Smaller models follow JSON instructions less reliably: compare the output with the hosted models.

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]
outputs = pipeline(messages, max_new_tokens=700)
print(outputs[0]["generated_text"][-1]["content"])

## Reasoning model

Reasoning models think before answering. Two cost controls in the OpenAI Responses API:

* `reasoning.effort`: `minimal | low | medium | high` - how much the model thinks.
* `text.verbosity`: `low | medium | high` - how long the final answer is.

Controlling both keeps **cost per resolved ticket** under control, an important metric once an
agent handles thousands of conversations a day.

In [ ]:
from openai import OpenAI
client = OpenAI()

resp_reasoning = client.responses.create(
    model="gpt-5",
    input=[{"role": "system", "content": system_prompt},
           {"role": "user", "content": user_prompt}],
    text={"format": {"type": "json_object"}, "verbosity": "low"},
    reasoning={"effort": "minimal"},
)
print(json.dumps(json.loads(resp_reasoning.output_text), indent=2, ensure_ascii=False))

## Discussion (MBA class)

1. Which model captured the **commitments** (72-hour replacement, Rs 3,000 credit, supervisor callback) most precisely?
2. Which one flagged the **risks** (social media, consumer court, Platinum churn)? Why do risks matter more than the summary?
3. Cost vs. quality: if NovaMart handles 50,000 calls a month, which model would you deploy for after-call notes, and why?
4. Data privacy: when is an open-weights model worth the extra engineering effort?